Installing dependencies

In [39]:
pip install langchain langchain-groq langchain-huggingface langchainhub -q


Note: you may need to restart the kernel to use updated packages.


In [40]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


Intialize openai llm with langchain

In [41]:

import os
os.environ["GROQ_API_KEY"] = os.getenv("api_key")

from langchain_groq import ChatGroq

llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature = 0.7)

print("Ready!")

Ready!


1-prompt templates

with templates:
"Translate '{word}' to {language}"

In [42]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    "Translate the word '{word}' to {language}.just give translation nothing else"
)

prompt = template.format(word = "hello" , language  = "French")
print(prompt)


Translate the word 'hello' to French.just give translation nothing else


In [43]:
response = llm.invoke(prompt)
print(response.content)

bonjour


In [44]:
for word , lang in [("hello","French"),("Goodbye","spanish"),("how are you","German")]:
    prompt = template.format(word =word,language = lang)
    response = llm.invoke(prompt)
    print(f"{word} in {lang} ->{response.content}")

hello in French ->bonjour
Goodbye in spanish ->Adiós
how are you in German ->Wie geht es dir?


prompt template using multiple templates


In [45]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    "explain {topic} to an {audience} in {sentences} sentences"
)

prompt = template.format(topic = "quantum computing" , audience = "5 year old", sentences = 5)

response = llm.invoke(prompt)
print(response.content)

Imagine a tiny, magical playground where the toys can be in many places at once, like a ball that can roll left, right, *and* stay still all at the same time. In a quantum computer, the tiny pieces called “qubits” are like those magical toys—they can be both on and off together, which lets them try lots of puzzles all at once. It’s like having a super‑fast superhero who can read many storybooks at the same time and find the best ending instantly. Because of this special magic, quantum computers can solve some really big problems much faster than ordinary computers. So, a quantum computer is a super‑clever, magical helper that can think about many possibilities all at once!


In [46]:
prompt = template.format(topic = "quantum computing" , audience = "50 year old", sentences = 5)

response = llm.invoke(prompt)
print(response.content)

Quantum computing uses the strange rules of quantum physics to process information in a way that ordinary computers can’t. Instead of bits that are either 0 or 1, quantum computers use “qubits” that can be 0, 1, or both at the same time, letting them explore many possibilities simultaneously. By linking qubits together through a property called entanglement, a change to one qubit instantly influences its partner, enabling massive parallel calculations. This means certain problems—like cracking complex codes, simulating molecules for drug discovery, or optimizing huge logistical networks—can be solved far faster than with classical machines. However, building reliable quantum computers is still very hard, so today they’re mostly experimental tools rather than everyday devices.


Using chains(temp+model)


In [47]:
chain = template | llm
response = chain.invoke({"topic":"quatum computing", "audience" : "5 year old", "sentences":5})
print(response.content)

Think of a regular computer like a line of tiny light switches that can only be **on** or **off** – like a tiny flashlight that’s either shining or not.  
A quantum computer uses teeny‑tiny things called **qubits** that can be **on, off, or both at the same time**, kind of like a magical coin that can land heads, tails, and a mix of both together.  
Because they can be in many states at once, a quantum computer can try lots of different answers all at the same time, like looking for a hidden treasure by checking many maps at once.  
When we finally look at the answer, the qubits “choose” one of the possibilities, just like when you finally open a surprise present.  
So a quantum computer is a super‑fast, super‑clever helper that can solve some puzzles much quicker than the ordinary on‑off computers we use every day.


2-Chat prompt Template ||
system - you - si behaviour ||
human - the user - actual question ||
ai - prev ai response



In [48]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages(
    [
        ("system","you are a {role}.Always respond in {tone} tone."),
        ("human","{question}"),
    ]
)

prompt = template.format_messages(
    role = "caring girlfriend" , tone = "sweat and caring" , question = "do you love me?"
)

RESPONSE = llm.invoke(prompt)
print(RESPONSE.content)

Of course I love you, sweetie! ❤️ Every moment with you feels like a little piece of sunshine, and I’m so grateful to have you in my life. You mean the world to me! 🌟💕


In [49]:
#chatpromptTemplate using chain

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system","you are a {role}.Always respond in {tone} tone."),
        ("human","{question}"),
    ]
)
llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature = 0.7)

chain = chat_template | llm

response = chain.invoke({"role":"friend , who talks very less", "tone":"friendly and caring", "question": "do you have a lighter?"})

print(response.content)

I’m sorry—I don’t have one. Stay safe!


PROMPT ENGINEERING TECHNIQUES

zero shot prompting 

In [50]:
from langchain_core.prompts import ChatPromptTemplate

zero_shot = ChatPromptTemplate.from_messages(
    [
        ("system","Classify the sentiment of the text as positive, negative or neutral.Only give the sentiment nothing else."),
        ("human","{text}"),
    ]
)

chain = zero_shot | llm

response = chain.invoke({"text": " i love this product! It has changed my life for the better"})
print(response.content)

positive


few shot prompting

give few example model learns from patterns

In [51]:
from langchain_core.prompts import ChatPromptTemplate

few_shot = ChatPromptTemplate.from_messages(
    [
        ("system","you classify customer reviews into categories : product , delivery,service,pricing."),
        ("human","I love this product! It has changed my life for the better."),
        ("ai","product"),
        ("human","support team very slow and unhelpful. I had to wait for 2 weeks to get a response."),
        ("ai","service"),
        ("human","The delivery was delayed by 3 days and the package arrived damaged."),
        ("ai","delivery"),
        ("human","The product is overpriced compared to similar items on the market."),
        ("ai","pricing"),
        ("human","{review}"),
    ]
)

chain = few_shot | llm

test_review = ["The product is great",
"but the delivery was delayed and the support team was unhelpful.",
"Overall, I am disappointed with the service and would not recommend this company to others.",
"However, the pricing is reasonable and the product itself is of good quality."]

for review in test_review:
    response = chain.invoke({"review" : review})
    print(response.content)

product
delivery
service
pricing


chain of thought prompting


In [52]:
#without cot 
simple= ChatPromptTemplate.from_messages([
    ("system","answer the question"),
    ("human","{question}"),
])

chain = simple| llm 

question = "A store has 45 apples . they sell 15 apples on Monday and 10 apples on Tuesday. How many apples are left in the store?"

response= chain.invoke({"question" : question})
print("without cot ->\n",response.content)

without cot ->
 The store started with 45 apples.

- Apples sold on Monday: 15  
- Apples sold on Tuesday: 10  

Total apples sold = 15 + 10 = **25**.

Remaining apples = 45 − 25 = **20**.

**There are 20 apples left in the store.**


In [53]:
with_cot = ChatPromptTemplate.from_messages([
    
    ("system","answer the question,Let's think step by step.show the work and then give the final answer"),
    ("human","{question}"),
])
chain = with_cot | llm
response= chain.invoke({"question" : question})
print("with cot ->\n",response.content)


with cot ->
 **Step‑by‑step calculation**

1. **Start with the initial number of apples:**  
   \(45\) apples.

2. **Subtract the apples sold on Monday:**  
   \(45 - 15 = 30\) apples remain after Monday.

3. **Subtract the apples sold on Tuesday:**  
   \(30 - 10 = 20\) apples remain after Tuesday.

**Final answer:**  
The store has **20 apples** left.


ROLE BASED PROMPTING

In [54]:
role_template = ChatPromptTemplate.from_messages([
    ("system", "you are {role}."),
    ("system", "respond from this perspective."),
    ("system", "use language and the examples appropriate for your role."),
    ("system", "always give answers in a short and effective manner"),
    ("human", "{question}"),
])

question = "Is there a scope of ai engineering as my career option?"

role = [
    "a senior ai engineer with 10 years of experience",
    "a college student who is interested in ai engineering",
    "a career counselor who has helped many students choose their career paths",
] 

chain = role_template | llm

for roles in role:
    response = chain.invoke({"role" : roles , "question":question})
    print("_______________________________________________________________________________")
    print(f"response from {roles}->",response.content)



_______________________________________________________________________________
response from a senior ai engineer with 10 years of experience-> Absolutely—AI engineering is one of the fastest‑growing tech careers today.

**Why it’s a strong option**
- **Demand:** Companies across every sector (finance, health, retail, autonomous systems, etc.) are hiring AI engineers to build, deploy, and maintain ML pipelines.
- **Compensation:** Salaries regularly top $150k – $250k + in major markets, with equity and bonuses for high‑impact projects.
- **Career ladder:** Junior → Mid‑level → Staff/Principal → AI Architect / Head of ML, plus pathways into product management or research leadership.
- **Skill relevance:** Expertise in data engineering, model development, MLOps, and responsible AI (fairness, privacy, governance) is increasingly non‑negotiable.

**Key skills to cultivate**
1. **Core ML:** Deep learning frameworks (PyTorch, TensorFlow), statistical modeling, and algorithmic intuition.
2. 

prompt chaining (multi step )